# OpenAI实现简历信息提取智能体

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("apikey.env")

DEEPSEEK_API = os.getenv("DEEPSEEK-API-KEY")
base_url = 'https://api.deepseek.com'
if DEEPSEEK_API:
    print("sucessfully got key!")
else:
    print("nedd API KEY")
chat_model = "deepseek-chat"


client = OpenAI(
    api_key = DEEPSEEK_API,
    base_url = base_url
)

sucessfully got key!


In [2]:
def get_completion(prompt):
    response = client.chat.completions.create(
        model=chat_model,  # 填写需要调用的模型名称
        messages=[
            {"role": "user", "content": prompt},
        ],
    )
    return response.choices[0].message.content
response = get_completion("你是谁？")
print(response)

你好！我是DeepSeek，由深度求索公司创造的AI助手！😊

我是一个纯文本模型，虽然不支持多模态识别功能，但我有文件上传功能，可以帮你处理图像、txt、pdf、ppt、word、excel等文件，从中读取文字信息进行分析处理。我完全免费使用，拥有128K的上下文长度，还支持联网搜索功能（需要你手动在Web/App中点开联网搜索按键）。

你可以通过官方应用商店下载我的App来使用。我很乐意为你解答问题、协助处理各种任务，无论是学习、工作还是日常生活中的疑问，我都会热情细致地帮你解决！

有什么我可以帮你的吗？随时问我哦！✨


In [3]:
from datetime import datetime, date
from typing import List, Optional
from pydantic import BaseModel, Field, field_validator, EmailStr, model_validator

首先定义一个pydantic类，定义提取简历的输出结构


In [ ]:
# 定义这个pydantic模型是关键的关键
class Resume(BaseModel):
    name: Optional[str] = Field(None, description="求职者姓名，如果没找到就置为空字符串")
    city: Optional[str] = Field(None, description="求职者居住地，如果没找到就置为空字符串")
    birthday: Optional[str] = Field(None, description="求职者生日，如果没找到就置为空字符串")
    phone: Optional[str] = Field(None, description="求职者手机号，如果没找到就置为空字符串")
    email: Optional[str] = Field(None, description="求职者邮箱，如果没找到就置为空字符串")
    education: Optional[List[str]] = Field(None, description="求职者教育背景")
    experience: Optional[List[str]] = Field(None, description="求职者工作或实习经历，如果没找到就置为空字符串")
    project: Optional[List[str]] = Field(None, description="求职者项目经历，如果没找到就置为空字符串")
    certificates: Optional[List[str]] = Field(None, description="求职者资格证书，如果没找到就置为空字符串")

    # 在 Pydantic 内部的类型转换（parsing）之前运行
    @field_validator("birthday", mode="before")
    def validate_and_convert_date(cls, raw_date):
        if raw_date is None:
            return None
        if isinstance(raw_date, str):
            # List of acceptable date formats
            date_formats = ['%d-%m-%Y', '%Y-%m-%d', '%d/%m/%Y', '%m-%d-%Y']
            for fmt in date_formats:
                try:
                    # Attempt to parse the date string with the current format
                    parsed_date = datetime.strptime(raw_date, fmt).date()
                    # Return the date in MM-DD-YYYY format as a string
                    return parsed_date.strftime('%m-%d-%Y')
                except ValueError:
                    continue  # Try the next format
            # If none of the formats match, raise an error
            raise ValueError(
                f"Invalid date format for 'consultation_date'. Expected one of: {', '.join(date_formats)}."
            )
        if isinstance(raw_date, date):
            # Convert date object to MM-DD-YYYY format
            return raw_date.strftime('%m-%d-%Y')

        raise ValueError(
            "Invalid type for 'consultation_date'. Must be a string or a date object."
        )

通过 `Resume().model_json_schema()` 得到一份json格式的schema

```bash
{
  "title": "Resume",
  "type": "object",
  "properties": {
    "name": {"type": "string"},
    "city": {"type": "string"},
    "education": {"type": "array", "items": {"type": "string"}}
  }
}
```



In [5]:
class ResumeOpenAI:
    def __init__(self):
        self.resume_profile = Resume()
        self.output_schema = self.resume_profile.model_json_schema()
        self.template = """
        You are an expert in analyzing resumes. Use the following JSON schema to extract relevant information:
        ```json
        {output_schema}
        ```json
        Extract the information from the following document and provide a structured JSON response strictly adhering to the schema above. 
        Please remove any ```json ``` characters from the output. Do not make up any information. If a field cannot be extracted, mark it as `n/a`.
        Document:
        ----------------
        {resume_content}
        ----------------
        """

    def create_prompt(self, output_schema, resume_content):
        return self.template.format(
            output_schema=output_schema,
            resume_content=resume_content
        )

    def run(self, resume_content):
        try:
            response = client.chat.completions.create(
                model=chat_model,
                # 不是所有模型都支持response_format，要看一下调用的模型是否支持这个参数
                # 千问、智谱的模型一般支持
                response_format={ "type": "json_object" },
                messages=[
                    {"role": "system", "content": "你是一位专业的简历信息提取专家。"},
                    {"role": "user", "content": self.create_prompt(self.output_schema, resume_content)}
                ],
            )

            result = response.choices[0].message.content
        except Exception as e:
            print(f"Error occurred: {e}")

        return result

resume_openai = ResumeOpenAI()

In [6]:
resume = """
姓名：张三 （Zhang San）
手 机：138-88O8-1234    邮箱：zhangsan#example,com  
居住地: 上海浦东新区||世纪大道888号  
生日: 1999年12月31日（公历） 
出生日期：31-12-99  

==========================
EDUCATION BACKGROUND
==========================
上海交通大学
Computer Science & Techology（貌似打错）  
2017.09 ~ 2021.06  

课程：Data Structre, Alogrithm, 线性代数, Opareting System

学历：本科

==========================
工作经验 / EXPERIENCE
==========================
Tencent Inc.
Internship — AI Engineer  
2020/07 ~ 2020/09  
主要负责：
- 图像分类模型调优（ResNet50）
- ⚙️ 数据清洗脚本开发（Python）
- 部署测试环境（Docker + Linux）

ByteD@nce — NLP算法实习生（2021.03—2021.09）
职责：
* 情感分析任务Fine-tune
* 数据标注与样本扩充

==========================
PROJECTS（项目经验）
==========================
Project: 图像识别增强系统 (2020.5 - 2020.8)
职责: 模型训练 & 性能优化  
成绩: Top-3 accuracy 97%

[Proj#2] 智能客服问答机器人（2021年）
描述: 使用BERT + Flask 实现FAQ问答接口（demo阶段）

==========================
CERTIFICATES & SKILLS
==========================
CET-6（550分）
国家计算机二级
💻 Skills：Python, PyTorch, SQL, Git, Linux, FastAPI

==========================
备注（Notes）
==========================
- 曾获2020年“创青春”创新大赛二等奖；
- 兴趣：编程、羽毛球、AI绘画；
- 出生日期格式在上面好像有点乱（呵呵）
- 一些乱码测试：ÂÎÄÑ—测试乱码字段—™®±∑
"""

In [7]:
rec_data = resume_openai.run(resume)
print(rec_data)

{
    "name": "张三 （Zhang San）",
    "city": "上海浦东新区||世纪大道888号",
    "birthday": "1999年12月31日（公历）",
    "phone": "138-88O8-1234",
    "email": "zhangsan#example,com",
    "education": ["上海交通大学 Computer Science & Techology（貌似打错） 2017.09 ~ 2021.06 课程：Data Structre, Alogrithm, 线性代数, Opareting System 学历：本科"],
    "experience": ["Tencent Inc. Internship — AI Engineer 2020/07 ~ 2020/09 主要负责：- 图像分类模型调优（ResNet50）- ⚙️ 数据清洗脚本开发（Python）- 部署测试环境（Docker + Linux）", "ByteD@nce — NLP算法实习生（2021.03—2021.09） 职责：* 情感分析任务Fine-tune * 数据标注与样本扩充"],
    "project": ["Project: 图像识别增强系统 (2020.5 - 2020.8) 职责: 模型训练 & 性能优化 成绩: Top-3 accuracy 97%", "[Proj#2] 智能客服问答机器人（2021年） 描述: 使用BERT + Flask 实现FAQ问答接口（demo阶段）"],
    "certificates": ["CET-6（550分）", "国家计算机二级", "💻 Skills：Python, PyTorch, SQL, Git, Linux, FastAPI"]
}
